# Vector Databases — First Contact

**Mental model:** instead of searching for exact matches, vector databases find things that are **SIMILAR**. You convert text, images, or any data into a list of numbers (a vector/embedding) that captures its meaning. Then you ask "find me the 5 vectors most similar to this one."

This powers semantic search, recommendations, and RAG pipelines. **pgvector** adds this capability directly to Postgres — no separate service needed.

## What makes vector search different

- **Embeddings** — convert data to dense numeric vectors that capture semantic meaning. "CPU spike" and "processor overload" land near each other in vector space even though they share no words.
- **Similarity search** — find nearest neighbors by cosine similarity or L2 distance, not exact string matching.
- **ANN** — Approximate Nearest Neighbor. For large datasets, exact nearest neighbor is too slow. ANN trades tiny accuracy loss for massive speed gain.
- **When to use** — semantic search, RAG pipelines, duplicate detection, anomaly detection, recommendation engines

In [ ]:
from pathlib import Path
import sys
for _candidate in [Path('_setup'), Path('Basics/Databases/_setup')]:
    if _candidate.exists():
        sys.path.insert(0, str(_candidate.resolve()))
        break

from db_connections import get_postgres_conn
import pandas as pd
import json

conn = get_postgres_conn()
cur = conn.cursor()

# Check pgvector extension is installed
cur.execute("SELECT extname, extversion FROM pg_extension WHERE extname = 'vector'")
row = cur.fetchone()
if row:
    print(f"pgvector installed: version {row[1]}")
else:
    print("pgvector not installed — run: CREATE EXTENSION vector;")
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    conn.commit()
    print("pgvector extension created.")

In [ ]:
# Real embeddings come from models like OpenAI or sentence-transformers
# For this notebook we generate random 384-dim vectors (same dim as
# sentence-transformers all-MiniLM-L6-v2) to demonstrate the mechanics.
# In Round 2 we will use real embeddings from alert text.

import numpy as np

# Create table for alert embeddings
cur.execute("DROP TABLE IF EXISTS telemetry.alert_embeddings")
cur.execute("""
    CREATE TABLE telemetry.alert_embeddings (
        alert_id    UUID PRIMARY KEY,
        endpoint_id UUID,
        message     TEXT,
        severity    TEXT,
        embedding   vector(384)
    )
""")
conn.commit()

# Load alerts from Postgres and generate fake embeddings
cur.execute("""
    SELECT alert_id, endpoint_id, message, severity
    FROM telemetry.alerts
    LIMIT 500
""")
alerts = cur.fetchall()

# Insert with random embeddings (normalized for cosine similarity)
insert_count = 0
for alert_id, endpoint_id, message, severity in alerts:
    vec = np.random.randn(384).astype(np.float32)
    vec = vec / np.linalg.norm(vec)  # normalize for cosine similarity
    vec_str = '[' + ','.join(f'{v:.6f}' for v in vec) + ']'
    cur.execute("""
        INSERT INTO telemetry.alert_embeddings
            (alert_id, endpoint_id, message, severity, embedding)
        VALUES (%s, %s, %s, %s, %s::vector)
        ON CONFLICT (alert_id) DO NOTHING
    """, (str(alert_id), str(endpoint_id), message, severity, vec_str))
    insert_count += 1

conn.commit()
print(f"Inserted {insert_count} alert embeddings (384-dim random vectors)")
print("Note: random vectors demonstrate mechanics — real embeddings come from text models")

In [ ]:
# Without an index, pgvector does exact nearest-neighbor (slow on large datasets)
# HNSW (Hierarchical Navigable Small World) is the ANN index — fast approximate search
cur.execute("""
    CREATE INDEX IF NOT EXISTS alert_embeddings_hnsw_idx
    ON telemetry.alert_embeddings
    USING hnsw (embedding vector_cosine_ops)
""")
conn.commit()
print("HNSW index created on alert_embeddings.embedding")
print("vector_cosine_ops = cosine similarity (best for normalized vectors)")
print("Alternatives: vector_l2_ops (L2 distance), vector_ip_ops (inner product)")

## 5 vector queries against the telemetry alerts

In [ ]:
# Simulate a search query — in production this would be
# the embedding of a user's search text
np.random.seed(42)
query_vec = np.random.randn(384).astype(np.float32)
query_vec = query_vec / np.linalg.norm(query_vec)
query_str = '[' + ','.join(f'{v:.6f}' for v in query_vec) + ']'

cur.execute("""
    SELECT
        message,
        severity,
        1 - (embedding <=> %s::vector) AS cosine_similarity
    FROM telemetry.alert_embeddings
    ORDER BY embedding <=> %s::vector
    LIMIT 5
""", (query_str, query_str))

rows = cur.fetchall()
df1 = pd.DataFrame(rows, columns=['message', 'severity', 'cosine_similarity'])
df1['cosine_similarity'] = df1['cosine_similarity'].round(4)
print("Top 5 most similar alerts to query vector:")
print(df1.to_string(index=False))

In [ ]:
# pgvector supports pre-filtering with WHERE before similarity search
# Find most similar alerts but only among critical severity
cur.execute("""
    SELECT
        message,
        severity,
        1 - (embedding <=> %s::vector) AS cosine_similarity
    FROM telemetry.alert_embeddings
    WHERE severity = 'critical'
    ORDER BY embedding <=> %s::vector
    LIMIT 5
""", (query_str, query_str))

rows = cur.fetchall()
df2 = pd.DataFrame(rows, columns=['message', 'severity', 'cosine_similarity'])
df2['cosine_similarity'] = df2['cosine_similarity'].round(4)
print("Top 5 most similar CRITICAL alerts:")
print(df2.to_string(index=False))
print("\nNote: WHERE clause filters BEFORE similarity ranking.")
print("This is pre-filtering — fast but may miss some relevant results.")

In [ ]:
# Find pairs of alerts that are very similar to each other
# Useful for deduplication and clustering
cur.execute("""
    SELECT
        a.message AS alert_a,
        b.message AS alert_b,
        1 - (a.embedding <=> b.embedding) AS similarity
    FROM telemetry.alert_embeddings a
    JOIN telemetry.alert_embeddings b
        ON a.alert_id < b.alert_id
    WHERE 1 - (a.embedding <=> b.embedding) > 0.95
    LIMIT 10
""")
rows = cur.fetchall()
if rows:
    df3 = pd.DataFrame(rows, columns=['alert_a', 'alert_b', 'similarity'])
    print(f"Near-duplicate alerts (similarity > 0.95): {len(df3)} pairs")
    print(df3.to_string(index=False))
else:
    print("No near-duplicate pairs found (expected with random vectors)")
    print("With real text embeddings, duplicate alert messages would cluster here")

In [ ]:
# Group alerts by how similar they are to the query
# Shows the distribution of the embedding space
cur.execute("""
    SELECT
        CASE
            WHEN 1 - (embedding <=> %s::vector) > 0.9  THEN 'very similar  (>0.9)'
            WHEN 1 - (embedding <=> %s::vector) > 0.7  THEN 'similar       (0.7-0.9)'
            WHEN 1 - (embedding <=> %s::vector) > 0.5  THEN 'somewhat      (0.5-0.7)'
            ELSE                                             'different     (<0.5)'
        END AS similarity_bucket,
        COUNT(*) AS count
    FROM telemetry.alert_embeddings
    GROUP BY 1
    ORDER BY 1
""", (query_str, query_str, query_str))

rows = cur.fetchall()
df4 = pd.DataFrame(rows, columns=['similarity_bucket', 'count'])
print("Distribution of similarity scores:")
print(df4.to_string(index=False))
print("\nWith random vectors, expect roughly uniform distribution.")
print("With real embeddings, most alerts cluster by category.")

In [ ]:
cur.execute("""
    EXPLAIN (FORMAT TEXT)
    SELECT message, 1 - (embedding <=> %s::vector) AS score
    FROM telemetry.alert_embeddings
    ORDER BY embedding <=> %s::vector
    LIMIT 5
""", (query_str, query_str))
plan = cur.fetchall()
print("Query plan for vector similarity search:")
for row in plan:
    print(row[0])
print("\nLook for 'Index Scan using alert_embeddings_hnsw_idx'")
print("That confirms the HNSW index is being used, not a sequential scan.")

## Exact search vs SQL — same question, different model

| SQL (Postgres) | pgvector |
|----------------|----------|
| `WHERE message LIKE '%CPU%'` | `ORDER BY embedding <=> query_vec LIMIT 5` |
| Exact string match only | Semantic similarity — finds related concepts |
| "CPU spike" ≠ "processor overload" | "CPU spike" ≈ "processor overload" |
| Fast with text index | Fast with HNSW index |
| No concept of "closeness" | Cosine similarity score 0.0–1.0 |
| `WHERE + AND` filters | `WHERE` pre-filter + similarity ranking |

## Key observations

- **`<=>` operator is cosine distance** (lower = more similar). Use `1 - (a <=> b)` to get cosine **similarity** (higher = more similar).
- **Random vectors demonstrate mechanics** but have no semantic meaning. In Round 2, real embeddings from alert message text will make "CPU spike" and "high processor load" land near each other.
- **HNSW vs IVFFlat** — HNSW is better for most use cases: faster queries, no training step needed, good recall. IVFFlat needs a training step (`VACUUM ANALYZE`) but uses less memory.
- **pgvector lives inside Postgres** — no separate vector service. You get ACID transactions, joins with regular tables, and standard Postgres backups all in one place.
- **Citi hook** — alert deduplication: when 50 endpoints emit the same "disk full" alert, real embeddings cluster them together so your on-call engineer sees 1 incident, not 50 pages.